In [10]:
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and

import argparse
import functools
import gc
import itertools
import logging
import math
import os
from distutils.util import strtobool
import random
import shutil
import warnings
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import torch.utils.checkpoint
import transformers
from accelerate import Accelerator
from accelerate.logging import get_logger
from accelerate.utils import (
    DistributedDataParallelKwargs,
    ProjectConfiguration,
    set_seed,
)
from huggingface_hub import create_repo, upload_folder
from huggingface_hub.utils import insecure_hashlib
from packaging import version
from PIL import Image
from PIL.ImageOps import exif_transpose
from torch.utils.data import Dataset
from torchvision import transforms
from tqdm.auto import tqdm
from transformers import AutoTokenizer, PretrainedConfig

import diffusers
from diffusers import (
    AutoencoderKL,
    DDPMScheduler,
    DPMSolverMultistepScheduler,
    StableDiffusionXLPipeline,
)

from diffusers.loaders import LoraLoaderMixin
from diffusers.optimization import get_scheduler
from diffusers.utils import check_min_version, is_wandb_available
from diffusers.utils.import_utils import is_xformers_available

from unziplora_unet.unziplora_linear_layer import UnZipLoRALinearLayer
from unziplora_unet.pipeline_stable_diffusion_xl import StableDiffusionXLUnZipLoRAPipeline, StableDiffusionXLSingleLoRAPipeline
from unziplora_unet.unet_2d_condition import UNet2DConditionModel
from unziplora_unet.utils import *

In [11]:
pretrained_model_name_or_path = '/home/xzh/xzh/pretrained/sd_xl_base_1.0'

In [12]:
vae_path = '/home/xzh/xzh/pretrained/sd_xl_base_1.0'
vae = AutoencoderKL.from_pretrained(
    pretrained_model_name_or_path,
    subfolder="vae",
    revision=None,
    torch_dtype=torch.float32,
)

In [13]:
pipeline = StableDiffusionXLSingleLoRAPipeline.from_pretrained(
    pretrained_model_name_or_path,
    vae=vae,
    revision= None,
    torch_dtype=torch.float32,
    max_rank=64,
    min_rank=32,
    alpha=1.0,
    branch="content",
    timestep_mode='priecewise'
)

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

/home/xzh/miniconda3/envs/unziplora/lib/python3.11/site-packages/diffusers/utils/outputs.py:63: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  torch.utils._pytree._register_pytree_node(


In [14]:
from unziplora_unet.singlelora import LoRACrossAttnProcessor, LoRALinearLayer
from safetensors.torch import load_file
import os

In [15]:
def _load_lora_state_dict(path: str):
    if os.path.isdir(path):
        preferred = os.path.join(path, "pytorch_lora_weights.safetensors")
        if os.path.isfile(preferred):
            return load_file(preferred)
        
        safetensors_files = sorted([f for f in os.listdir(path) if f.endswith(".safetensors")])
        if not safetensors_files:
            raise ValueError(f"No .safetensors file found in {path}")
        return load_file(os.path.join(path, safetensors_files[0]))
    return load_file(path)

In [16]:
lora_path = '/home/xzh/xzh/UnZipLoRA-heatmap/models/anime_cat/anime_cat_content/pytorch_lora_weights.safetensors'
lora_state_dict = _load_lora_state_dict(lora_path)

In [17]:
lora_state_dict

{'unet.unet.down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_k.lora.down.weight': tensor([[-0., -0., -0.,  ..., 0., -0., -0.],
         [0., -0., -0.,  ..., -0., 0., 0.],
         [-0., 0., -0.,  ..., 0., 0., -0.],
         ...,
         [0., 0., 0.,  ..., -0., -0., -0.],
         [-0., 0., 0.,  ..., 0., 0., 0.],
         [-0., -0., -0.,  ..., -0., 0., -0.]]),
 'unet.unet.down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_k.lora.up.weight': tensor([[-0.0075, -0.0378, -0.0700,  ...,  0.0706, -0.0067,  0.0417],
         [ 0.0296, -0.0308,  0.0615,  ...,  0.0119,  0.0074, -0.0136],
         [-0.0093,  0.0304, -0.0294,  ...,  0.0557,  0.0592,  0.0289],
         ...,
         [-0.0148,  0.0013,  0.0037,  ..., -0.0067,  0.0248,  0.0524],
         [ 0.0348, -0.0175,  0.0253,  ..., -0.0057, -0.0459, -0.0155],
         [ 0.0407,  0.0592, -0.0298,  ..., -0.0453,  0.0134, -0.0818]]),
 'unet.unet.down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_out.0.lora.down.weight': tensor

In [ ]:
for name, processor in pipeline.unet.attn_processors.items():
    name_pre = '.'.join(name.split('.')[:-1])
    lora_prefix = f"unet.unet.{name_pre}"
    base_prefix = f"unet.{name_pre}"
    print(name_pre)
    
    print(base_prefix)
    # print:           unet.down_blocks.1.attentions.0.transformer_blocks.0.attn1
    # base_state_dict: unet.down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_q.lora.up.base_content
    
    print(lora_prefix)
    # print:           unet.unet.down_blocks.1.attentions.0.transformer_blocks.0.attn1
    # lora_state_dict: unet.unet.down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_out.0.lora.down.weight
    print()

down_blocks.1.attentions.0.transformer_blocks.0.attn1
unet.down_blocks.1.attentions.0.transformer_blocks.0.attn1
unet.unet.down_blocks.1.attentions.0.transformer_blocks.0.attn1

down_blocks.1.attentions.0.transformer_blocks.0.attn2
unet.down_blocks.1.attentions.0.transformer_blocks.0.attn2
unet.unet.down_blocks.1.attentions.0.transformer_blocks.0.attn2

down_blocks.1.attentions.0.transformer_blocks.1.attn1
unet.down_blocks.1.attentions.0.transformer_blocks.1.attn1
unet.unet.down_blocks.1.attentions.0.transformer_blocks.1.attn1

down_blocks.1.attentions.0.transformer_blocks.1.attn2
unet.down_blocks.1.attentions.0.transformer_blocks.1.attn2
unet.unet.down_blocks.1.attentions.0.transformer_blocks.1.attn2

down_blocks.1.attentions.1.transformer_blocks.0.attn1
unet.down_blocks.1.attentions.1.transformer_blocks.0.attn1
unet.unet.down_blocks.1.attentions.1.transformer_blocks.0.attn1

down_blocks.1.attentions.1.transformer_blocks.0.attn2
unet.down_blocks.1.attentions.1.transformer_blocks.0.att

In [33]:
base_weight_path = '/home/xzh/xzh/UnZipLoRA-heatmap/models/anime_cat/anime_cat_base_weight_content.pth'
base_state_dict = torch.load(base_weight_path, map_location="cpu")
base_state_dict

{'unet.down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_q.lora.up.base_content': tensor([[-0.0520,  0.0141,  0.0440,  ..., -0.0386, -0.0692,  0.0522],
         [ 0.0010, -0.0147,  0.0174,  ..., -0.0031,  0.0715, -0.0431],
         [-0.0230,  0.0081,  0.0038,  ..., -0.0201,  0.0509,  0.0641],
         ...,
         [ 0.0376, -0.0551, -0.0375,  ..., -0.0064,  0.0087,  0.0158],
         [ 0.0105,  0.0253,  0.0554,  ...,  0.0395,  0.0310,  0.0145],
         [ 0.0694, -0.0352,  0.0218,  ..., -0.0276, -0.0095, -0.0410]]),
 'unet.down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_q.lora.down.base_content': tensor([[0., -0., 0.,  ..., 0., -0., 0.],
         [0., 0., 0.,  ..., -0., 0., -0.],
         [-0., -0., -0.,  ..., 0., -0., -0.],
         ...,
         [-0., 0., 0.,  ..., -0., -0., 0.],
         [0., -0., 0.,  ..., 0., -0., 0.],
         [-0., 0., -0.,  ..., 0., -0., 0.]]),
 'unet.down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_k.lora.up.base_content': tensor([[-0.

In [34]:
merges_weight_path = '/home/xzh/xzh/UnZipLoRA-heatmap/models/anime_cat/anime_cat_merger_content.pth'
merges_state_dict = torch.load(merges_weight_path, map_location="cpu")
merges_state_dict

{'unet.down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_q.lora.merge_content': Parameter containing:
 tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]),
 'unet.down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_k.lora.merge_content': Parameter containing:
 tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]),
 'unet.down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_v.lora.merge_content': Parameter containing:
 tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
    